# **TTS Системы**

Вам предстоит поработать с TTS системами.

Возьмите небольшой (3-5 предложений) текстовый отрывок из вашего любимого фильма, книги или стихотворения на английском языке.

Озвучьте его с помощью одного (или более, если будет желание) из API с бесплатными генерациями:
https://zvukogram.com/
https://rapidapi.com/rahilkhan224/api/text-to-speech-neural-google/pricing
https://rapidapi.com/cloudlabs-dev/api/cloudlabs-text-to-speech/pricing

(Можете найти самостоятельно те API, что понравятся больше)

Затем вам нужно озвучить тот же фрагмент с помощью модели Tortoise TTS. Можете взять один из готовых голосов в репозитории, либо получить похожий на вашего любимого персонажа голос (помните о лицензиях и авторском праве).

В репозитории Tortoise TTS:
https://github.com/neonbjb/tortoise-tts

В репозитории есть как ipynb для работы, так и инструкции по получению похожего голоса.
Запускать рекомендую в Google Colab, Tortoise требователен к железу.

Результаты следует оформить в .ipynb. Прикрепите туда текст и ваши аудио результаты, так же кратко опишите ваше мнение о сравнении моделей. Опишите, какой результат воспринимается более качественным и ближе к настоящему человеку.



Текст (цитата из Форсажа)



> Money will come and go.  -.We all know that.  -.The most important thing in life will always be the people in this room. Right here,  -.right now.



In [ ]:
# Проверяем, что GPU активен
!nvidia-smi


Tue Oct  7 10:47:36 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8             10W /   70W |       2MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
!pip install gTTS

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 7.5 MB/s eta 0:00:00
  Attempting uninstall: click
    Found existing installation: click 8.3.0
    Uninstalling click-8.3.0:
      Successfully uninstalled click-8.3.0


In [34]:
text = "Money will come and go... We all know that... The most important thing in life will always be the people in this room... Right here, ... right now."

In [31]:
from gtts import gTTS
from IPython.display import Audio

tts = gTTS(text=text, lang='en')
tts.save("speech.mp3")
Audio("speech.mp3")

In [7]:
!pip install edge-tts --quiet

In [19]:
!edge-tts --list-voices > voices.txt

In [35]:
import asyncio
import edge_tts
from IPython.display import Audio

voice = "en-US-OnyxTurboMultilingualNeural"  # мужской
output_path = "edge_output.mp3"

async def generate_tts():
    communicate = edge_tts.Communicate(
        text=text,
        voice=voice,
        rate="-25%"   # примерно 0.75x скорости
     )
    await communicate.save(output_path)

await generate_tts()
Audio(output_path)


На мой взгляд более походе на человеческий голос получилось сделать, используя edge-tts


## 1. **gTTS (Google Text-to-Speech)**

**Тип:** облачная (не open-source) TTS, обёртка над API Google Translate.
**Уровень сложности:** минимальный (нет нейросетевого обучения локально).

### Архитектура:

* gTTS сам по себе **не содержит модели** — это просто **обёртка над Google Translate TTS endpoint**.
* Внутри Google использует **WaveNet** и **Tacotron 2** (в разных сервисах, например, Cloud TTS и Translate TTS).
* В бесплатной версии (`gTTS`) используется **упрощённый Tacotron-like pipeline**:

  * Text normalization → Phoneme conversion → Mel-spectrogram generation (Tacotron 2-подобная seq2seq модель)
  * Mel-spectrogram → аудио (вокодер на основе WaveRNN или WaveNet)
* Всё это выполняется **на стороне Google**, а ты просто получаешь MP3-файл.

### Основные параметры (доступные пользователю):

| Параметр | Описание                                          |
| -------- | ------------------------------------------------- |
| `text`   | Текст для озвучки                                 |
| `lang`   | Язык (например, `'en'`, `'ru'`, `'fr'`)           |
| `tld`    | Акцент / регион (например, `'co.uk'`, `'com.au'`) |
| `slow`   | Медленная речь                                    |
| —        | нет контроля тембра, пола, стиля                |

---

## 2. **Edge-TTS (Microsoft Neural TTS / Azure Speech)**

**Тип:** облачный API Microsoft Azure, но есть open интерфейс `edge-tts` (всё работает через WebSocket).
**Уровень сложности:** средний (модель готовая, но нейросетевая).

### Архитектура:

Edge TTS использует **Unified Neural TTS Architecture** от Microsoft — гибрид Tacotron + FastSpeech + нейронный вокодер (HiFi-GAN).

#### Общая схема:

```
Text → Phoneme Encoder → Duration Predictor → Mel Spectrogram → Neural Vocoder (HiFi-GAN)
```

**Ключевые компоненты:**

1. **Text Frontend:** нормализует и преобразует текст в фонемы.
2. **Encoder-Decoder (FastSpeech2-подобный):**

   * Duration Predictor (длина фонем)
   * Pitch & Energy Predictor
3. **Neural Vocoder:** HiFi-GAN или WaveGlow.
4. **Style embeddings:** специальные вектора для управления стилем (нейтральный, радостный, рекламный и т.д.)
5. **Voice Embeddings:** для каждого синтетического спикера (например, “GuyNeural”, “JennyNeural”, “DmitryNeural”).

### Основные параметры (в `edge-tts`):

| Параметр      | Пример                 | Описание                    |
| ------------- | ---------------------- | --------------------------- |
| `voice`       | `"en-US-GuyNeural"`    | выбор спикера               |
| `rate`        | `"-20%"`               | скорость речи               |
| `volume`      | `"+5%"`                | громкость                   |
| `pitch`       | `"-10Hz"` или `"+2st"` | высота голоса               |
| `style`       | `"cheerful"`, `"sad"`  | стиль (не у всех голосов)   |
| `styledegree` | `"1"`–`"2"`            | степень выраженности эмоции |
| `text`        | —                      | текст                       |

---

## 3. **Tortoise TTS**

**Тип:** полностью нейросетевая, open-source модель.
**Уровень сложности:** высокий (модель огромная, требует GPU и VRAM > 10 ГБ).

### Архитектура:

Tortoise объединяет несколько подходов:

1. **Text Encoder (CLIP-style transformer)** — превращает текст в эмбеддинги.
2. **Voice Conditioning Encoder** — анализирует референс-запись голоса и создаёт эмбеддинг говорящего.
3. **Diffusion Decoder (VAE + diffusion-like model)** — генерирует аудио из текстовых и голосовых эмбеддингов.
4. **Vocoder (HiFi-GAN)** — восстанавливает финальный аудиосигнал.

#### Схема:

```
Text → Text Encoder
Voice Sample → Voice Encoder
↓
[Concatenate Embeddings]
↓
Diffusion Decoder → Mel-Spectrogram → HiFi-GAN Vocoder → Audio
```

* Модель использует **автогенерацию нескольких вариантов аудио** (sampling → ranking по quality model).
* Выход максимально реалистичный, но **очень медленный** (30–60 сек/фраза на GPU).

### Основные параметры:

| Параметр                     | Описание                                |
| ---------------------------- | --------------------------------------- |
| `text`                       | текст                                   |
| `voice_samples`              | примеры голоса для клонирования         |
| `preset`                     | "fast", "high_quality", "ultra_fast"    |
| `cvvp_amount`                | вес проверки согласованности голоса     |
| `temperature`                | степень случайности                     |
| `num_autoregressive_samples` | количество проб для отбора              |
| `diffusion_iterations`       | глубина диффузии (качество vs скорость) |

---
